<a href="https://colab.research.google.com/github/202503660-ai/NLP_for_SocialScience/blob/Zero-Shot-Performance-and-Limitations-Analysis-of-Korean-Sentiment-Recognition-in-Multilingual-LLM/model_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm # 진행상황 표시를 위한 라이브러리 추가

# ==========================================
# 사용자 설정 변수 (이 부분을 수정하여 사용하세요)
# ==========================================

# 1. 사용할 허깅페이스 모델 이름
MODEL_NAME = "beomi/KoAlpaca-Polyglot-5.8B"

# 2. 감성분석 대상 텍스트가 있는 데이터의 컬럼명
INPUT_TEXT_COLUMN = "텍스트"

# 3. 테스크 텍스트 (감정분석 지시 / 프롬프트)
TASK_TEXT = """다음 문장의 감정을 분석하여 주어진 감정_대분류(기쁨, 슬픔, 분노, 불안, 당황, 상처) 중 하나만 답변하세요.
문장: {text}
감정:"""

# 4. 정답 라벨이 있는 컬럼명
LABEL_COLUMN = "감정_대분류"

# 5. 결과 저장 엑셀 파일 이름 설정
OUTPUT_EXCEL_PATH = "result_sentiment.xlsx"

# ==========================================
# 실행부 (이미 메모리에 'df_raw'라는 데이터프레임이 존재한다고 가정합니다)
# ==========================================

print(f"[{MODEL_NAME}] 모델을 로드 중입니다. (모델 크기에 따라 시간이 걸릴 수 있습니다...)")

# GPU 사용 가능 여부 확인
device = 0 if torch.cuda.is_available() else -1

try:
    # 모델 로드 (텍스트 생성 파이프라인)
    generator = pipeline(
        "text-generation",
        model=MODEL_NAME,
        device=device,
        torch_dtype=torch.float16 if device == 0 else torch.float32,
        device_map="auto" if device == 0 else None
    )
except Exception as e:
    print(f"모델 로드 중 오류가 발생했습니다. 모델 이름을 확인해주세요.\n에러: {e}")
    generator = None

if generator is not None:
    # 필수 컬럼 존재 여부 확인
    if INPUT_TEXT_COLUMN not in df_raw.columns:
        print(f"오류: 입력 텍스트 컬럼 '{INPUT_TEXT_COLUMN}'이(가) df_raw 데이터에 없습니다.")
    elif LABEL_COLUMN not in df_raw.columns:
        print(f"오류: 정답 라벨 컬럼 '{LABEL_COLUMN}'이(가) df_raw 데이터에 없습니다.")
    else:
        predictions = []

        print(f"제로샷 감성 분석을 시작합니다. (총 {len(df_raw)}개 데이터)")

        # tqdm을 이용한 진행 상황 표시 바 생성
        for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc="감성 분석 진행률"):
            text = str(row[INPUT_TEXT_COLUMN])

            # 결측치 처리
            if pd.isna(row[INPUT_TEXT_COLUMN]) or text.strip() == "":
                predictions.append("결측")
                continue

            # 프롬프트 생성
            prompt = TASK_TEXT.format(text=text)

            # 모델 추론
            try:
                result = generator(
                    prompt,
                    max_new_tokens=10,
                    num_return_sequences=1,
                    pad_token_id=generator.tokenizer.eos_token_id,
                    truncation=True
                )

                # 생성된 텍스트에서 모델의 답변만 추출
                generated_text = result[0]['generated_text']
                answer = generated_text[len(prompt):].strip().split('\n')[0]
                predictions.append(answer)
            except Exception as e:
                # 에러 로그 출력 (tqdm 바를 깨뜨리지 않기 위해 간단히 출력)
                print(f"\n{idx}번째 데이터 처리 중 오류: {e}")
                predictions.append("오류")

        # df_raw 데이터프레임에 예측 결과 컬럼 직접 추가
        df_raw['예측_감정'] = predictions

        # 정답/오답 판별 함수
        def check_correctness(row):
            pred_label = str(row['예측_감정']).strip()
            if pred_label in ["결측", "오류", ""]:
                return '판별불가'

            true_label = str(row[LABEL_COLUMN]).strip()

            # 예측된 문자열 안에 정답 라벨이 포함되어 있으면 정답으로 간주
            if true_label in pred_label:
                return '정답'
            else:
                return '오답'

        # 정답_여부 컬럼 추가
        df_raw['정답_여부'] = df_raw.apply(check_correctness, axis=1)

        # 정답률 계산
        correct_count = (df_raw['정답_여부'] == '정답').sum()
        valid_count = (df_raw['정답_여부'] != '판별불가').sum()
        accuracy = (correct_count / valid_count * 100) if valid_count > 0 else 0

        print(f"\n분석 완료! 총 {valid_count}건 중 {correct_count}건 정답 (정답률: {accuracy:.2f}%)")
        print("df_raw에 '예측_감정'과 '정답_여부' 컬럼이 성공적으로 추가되었습니다.")

        # ==========================================
        # 엑셀 파일로 결과 저장
        # ==========================================
        print(f"\n작업 결과를 '{OUTPUT_EXCEL_PATH}' 파일로 저장합니다...")
        try:
            df_raw.to_excel(OUTPUT_EXCEL_PATH, index=False)
            print(f"엑셀 저장 완료! (저장 위치: 현재 폴더의 {OUTPUT_EXCEL_PATH})")
        except Exception as e:
            print(f"엑셀 저장 중 오류가 발생했습니다: {e}")